# pysystemtrade 交互式学习笔记

> 跟随 `docs/introduction.md` 与 `examples/introduction/` 的路径，通过实际运行代码重新吃透这个项目。

**核心公式：一个交易系统 = 数据 (data) + 若干处理阶段 (stages) + 配置 (config)**

```mermaid
flowchart LR
    A[原始数据 data] --> B[Rules 交易规则]
    B --> C[ForecastScaleCap 缩放/截顶]
    C --> D[ForecastCombine 预测组合]
    D --> E[PositionSizing 头寸规模]
    E --> F[Portfolios 组合构建]
    F --> G[Account 盈亏核算]
    H[Config YAML/dict] -.控制各阶段行为.-> B
    H -.-> C
    H -.-> D
    H -.-> E
    H -.-> F
```

**三个关键设计模式（贯穿全项目）：**
1. **阶段名固定**：无论类/实例叫什么，访问永远是 `system.rules`、`system.combForecast`、`system.positionSize` 等
2. **TradingRule 三要素**：`function` + `data`（类似 *args）+ `other_args`（类似 **kwargs）
3. **Config 驱动**：生产环境用 YAML 配置控制系统行为，而不是散落在脚本里

注意：你看到的结果会与文档中略有不同，因为 CSV 数据已更新到更近的日期。

## 1. 加载模拟数据与探索合约信息

`data` 对象是整个系统的数据源。这里使用预烘焙的 CSV 数据 `csvFuturesSimData`。

**继承链**：`csvFuturesSimData`（数据特定）→ `FuturesData`（资产类别特定）→ `simData`（通用）。以后换成 IB 实时数据时，只需替换最左端的类，上层逻辑完全不变。

In [ ]:
from sysdata.sim.csv_futures_sim_data import csvFuturesSimData

data = csvFuturesSimData()
print(data)
print(data.get_instrument_list())

In [ ]:
# 合约元数据：点值、币种、资产类别、滑点、佣金假设
# 这些信息在之后的头寸规模和成本计算中都会用到
data.get_instrument_object_with_meta_data("MUMMY")

In [ ]:
# data 对象有"类字典"行为（但并不真的继承 dict）
print(data.keys())          # 等价于 get_instrument_list()
print(data["VIX"].tail(3))  # 等价于 get_instrument_price("VIX")

## 2. 获取价格与 Carry 数据

`get_raw_price()` 返回的是**回调整合 (back-adjusted) 价格**——用 panama 方法把相邻合约拼接起来的连续价格序列，而不是某个真实合约的价格。原因：

- 真实合约价格在不同合约间有跳变，用它算波动率会失真
- 拼接价格能捕捉**展期收益 (rolldown)**，参考 [Rob 的博客](https://qoppac.blogspot.com/2015/05/systems-building-futures-rolling.html)

期货特有的另一类数据是 **carry 数据**（实现 carry 规则所需）：

In [ ]:
data.get_raw_price("SOFR").tail(5)

In [ ]:
# carry 数据四列含义：
#   PRICE          当前交易合约的拼接价格
#   CARRY          用于计算 carry 的远月合约价格
#   CARRY_CONTRACT 远月合约月份（如 201903 = 2019年3月）
#   PRICE_CONTRACT 当前定价合约月份
data.get_instrument_raw_carry_data("SOFR").tail(6)

## 3. 手动实现 EWMAC 交易规则

**EWMAC**（Exponentially Weighted Moving Average Crossover）是全书最经典的趋势规则：

$$\text{forecast} = \frac{\text{EWMA}_{\text{fast}}(P) - \text{EWMA}_{\text{slow}}(P)}{\sigma(\Delta P)}$$

- 分子：快慢指数均线之差 → 趋势方向与强度
- 分母：波动率归一化 → 让不同波动率的品种产生的信号可比较

**关键点**：必须用拼接价格（而不是实际交易合约的价格），否则合约切换时波动率会跳变，且会丢失展期收益。

In [ ]:
import pandas as pd
from sysquant.estimators.vol import robust_vol_calc


def calc_ewmac_forecast(price, Lfast, Lslow=None):
    """
    计算 EWMAC 交易规则的预测值。

    price: 拼接后的价格序列（back-adjusted）
    Lfast: 快 EWMA 的 span；Lslow 缺省为 4 * Lfast
    """
    price = price.resample("1B").last()  # 重采样到工作日，取最后一个价
    if Lslow is None:
        Lslow = 4 * Lfast

    # 直接用 span，无需算衰减参数
    fast_ewma = price.ewm(span=Lfast).mean()
    slow_ewma = price.ewm(span=Lslow).mean()
    raw_ewmac = fast_ewma - slow_ewma

    vol = robust_vol_calc(price.diff())  # 稳健波动率（指数加权，抗离群值）
    return raw_ewmac / vol

In [ ]:
%matplotlib inline

instrument_code = "SOFR"
price = data.daily_prices(instrument_code)
ewmac = calc_ewmac_forecast(price, 32, 128)
print(ewmac.tail(5))
ewmac.plot(figsize=(12, 4), title=f"EWMAC(32,128) forecast - {instrument_code}")

## 4. 单规则盈亏分析（account_forecast）

`pandl_for_instrument_forecast` 把 forecast 直接当作头寸来结算盈亏（此处不含成本、未做缩放）。

返回的 `account` 对象**继承自 pandas DataFrame**，并附加了大量统计方法（`syscore.pandas.strategy_functions` 中定义）。

重点指标速查：
- `sharpe`：年化夏普比率
- `calmar`：年化收益 / 最大回撤类指标
- `hitrate`：胜率
- `t_stat` / `p_value`：统计显著性

In [ ]:
from systems.accounts.account_forecast import pandl_for_instrument_forecast

account = pandl_for_instrument_forecast(forecast=ewmac, price=price)
account.percent.stats()

In [ ]:
import syscore.pandas.strategy_functions  # 注册 .sharpe() 等扩展方法

print("年化夏普:", account.sharpe())
account.curve().plot(figsize=(12, 4), title="Cumulative % curve")   # 累计净值曲线
account.drawdown().plot(figsize=(12, 3), title="Drawdown (%)")      # 回撤曲线
# 其他可用：account.percent / account.weekly / account.monthly / account.gross.ann_mean()

## 5. 构建最小系统：Rules + System

单个脚本能做的有限。真正的威力在于**系统 (System)**：由 data + 若干 stage + 可选 config 组成。

完整阶段流水线：
1. RawData（原始数据预处理）
2. Rules（交易规则 → 原始预测）
3. ForecastScaleCap（缩放 + 截顶）
4. ForecastCombine（多规则组合）
5. PositionSizing（头寸规模）
6. Portfolios（组合构建）
7. Account（盈亏核算）

先用最简单的系统——只有一个 Rules 阶段。注意使用项目内置的 `ewmac_forecast_with_defaults`（带默认参数版本）。

In [ ]:
from systems.provided.rules.ewmac import ewmac_forecast_with_defaults as ewmac
from systems.forecasting import Rules
from systems.basesystem import System

# 方式一：直接传函数，规则自动命名为 rule0（不好认）
my_rules = Rules(ewmac)
print(my_rules.trading_rules())

# 方式二：传 dict，规则名一目了然
my_rules = Rules(dict(ewmac=ewmac))
print(my_rules.trading_rules())

In [ ]:
my_system = System([my_rules], data)
print(my_system)

# 通用模式：system.<stage名>.get_something(...)
# 结果与第 3 节手动计算完全一致——但现在有了可扩展的框架
my_system.rules.get_raw_forecast("SOFR", "ewmac").tail(5)

## 6. TradingRule 的多种定义方式与多规则变体

一个 `TradingRule` 包含 3 个元素：
- `function`：规则函数
- `data`：函数需要的数据列表（类似 *args，缺省时默认传日价格）
- `other_args`：其余参数 dict（类似 **kwargs）

**三种等价定义方式**：直接传函数 / 三元组 tuple / dict。用不同参数定义同一个规则的多个"风味"：

In [ ]:
from systems.trading_rules import TradingRule

ewmac_rule = TradingRule(ewmac)                                        # 最简形式
ewmac_8 = TradingRule((ewmac, [], dict(Lfast=8, Lslow=32)))            # tuple 形式
ewmac_32 = TradingRule(dict(function=ewmac,
                            other_args=dict(Lfast=32, Lslow=128)))     # dict 形式
print(ewmac_32)

my_rules = Rules(dict(ewmac8=ewmac_8, ewmac32=ewmac_32))
my_system = System([my_rules], data)

# ewmac32 参数等于默认值，结果应与第 5 节一致
my_system.rules.get_raw_forecast("SOFR", "ewmac32").tail(5)

## 7. Config 配置对象与 YAML 加载

**Config** 是控制各阶段行为的核心。三种创建方式：
1. `Config()` 空对象，之后逐个设属性
2. `Config(dict(...))` 从字典
3. `Config("systems.provided.example.simplesystemconfig.yaml")` 从 YAML（Python 风格路径引用，不写文件名）

**YAML 中的关键机制**：函数和数据源都用**字符串路径**指定（如 `systems.provided.rules.ewmac.ewmac_forecast_with_defaults`、`data.daily_prices`），系统启动时解析。这样纯文本配置就能完整描述一个系统。

In [ ]:
from sysdata.config.configdata import Config

# 方式一：空 Config + 属性注入。注意这里 Rules() 是空实例，规则由 config 提供
my_config = Config()
empty_rules = Rules()
my_config.trading_rules = dict(ewmac8=ewmac_8, ewmac32=ewmac_32)
my_system = System([empty_rules], data, my_config)
print(my_system.rules.get_raw_forecast("SOFR", "ewmac8").tail(3))

# 注意：如果同时把规则传给 Rules() 构造器和 config，只有前者生效

In [ ]:
# 方式三：从 YAML 文件加载（生产环境的标准做法）
yaml_config = Config("systems.provided.example.simplesystemconfig.yaml")
print(yaml_config)

# 对照打开 systems/provided/example/simplesystemconfig.yaml 查看：
# trading rule 的 function 是字符串路径；ewmac8 显式指定了 data: data.daily_prices（即默认值）

## 8. 预测缩放与截断（ForecastScaleCap）

原始 EWMAC 预测没有统一量纲。目标：**平均绝对值 = 10**，并**截顶到 ±20**（防止极端信号支配仓位）。

两种方式得到缩放因子 (forecast scalar)：
- **滚动样本外估计**：`use_forecast_scale_estimates=True`（默认跨品种池化估计，更稳健）
- **固定值**：`forecast_scalars=dict(...)`（可用书中附录 B 的经验值）

注意：传入 `System([...])` 的阶段列表**顺序无关**。

In [ ]:
from systems.forecast_scale_cap import ForecastScaleCap

fcs = ForecastScaleCap()

# 方式一：滚动样本外估计（跨品种池化）
my_config.instruments = ["SOFR", "US10", "CORN", "SP500_micro"]
my_config.use_forecast_scale_estimates = True

my_system = System([fcs, my_rules], data, my_config)
print(my_system.forecastScaleCap.get_forecast_scalar("SOFR", "ewmac32").tail(5))

In [ ]:
# 方式二：固定缩放因子（书中附录 B 的经验值）
my_config.forecast_scalars = dict(ewmac8=5.3, ewmac32=2.65)
my_config.use_forecast_scale_estimates = False

my_system = System([fcs, empty_rules], data, my_config)
print("scalar:", my_system.forecastScaleCap.get_forecast_scalar("SOFR", "ewmac32"))

# 截顶后的预测（默认 cap=20，定义在系统默认值文件）
my_system.forecastScaleCap.get_capped_forecast("SOFR", "ewmac32").tail(5)

## 9. 预测组合与多样化乘数（ForecastCombine）

多个规则变体需要组合。组合预测公式：

$$\text{combined} = \text{FDM} \times \sum_i w_i \cdot \text{forecast}_i$$

- $w_i$：预测权重 (forecast weights)
- **FDM**（forecast diversification multiplier）：多样化乘数，补偿权重分散造成的信号强度损失

三种方式：默认等权 + FDM=1（会打 WARNING）/ 滚动估计 / 固定值。

> **估计权重时需要 Account 阶段**：因为权重估计依赖各规则的历史表现，而历史表现需要 PositionSizing 和 RawData 才能算出来。这体现了阶段间的依赖关系。

In [ ]:
from systems.forecast_combine import ForecastCombine

combiner = ForecastCombine()
my_system = System([fcs, empty_rules, combiner], data, my_config)

# 未配置权重时：等权 0.5/0.5 + FDM=1（观察 WARNING 输出）
print(my_system.combForecast.get_forecast_weights("SOFR").tail(3))
print(my_system.combForecast.get_forecast_diversification_multiplier("SOFR").tail(3))

In [ ]:
# 固定权重 + FDM（实用起步方案）
my_config.forecast_weights = dict(ewmac8=0.5, ewmac32=0.5)
my_config.forecast_div_multiplier = 1.1
my_config.use_forecast_weight_estimates = False
my_config.use_forecast_div_mult_estimates = False

my_system = System([fcs, empty_rules, combiner], data, my_config)
my_system.combForecast.get_combined_forecast("SOFR").tail(5)

## 10. 头寸规模与波动目标（PositionSizing）

子系统头寸公式（对应书中第 9、10 章）：

$$\text{position} = \frac{\text{forecast}}{10} \times \frac{\text{capital} \times \text{vol\_target}}{\sigma_{\text{instrument}} \times \text{point\_size}}$$

直觉：**预测越强仓位越大；品种波动越大仓位越小；目标风险越高整体仓位越大**。这里新增 `RawData` 阶段（提供波动率等计算所需的原始数据加工）。

In [ ]:
from systems.rawdata import RawData
from systems.positionsizing import PositionSizing

raw_data = RawData()
position_size = PositionSizing()

my_config.percentage_vol_target = 25        # 年化波动目标 25%
my_config.notional_trading_capital = 500000 # 名义本金 50 万
my_config.base_currency = "GBP"             # 基础货币

my_system = System([fcs, empty_rules, combiner, position_size, raw_data], data, my_config)
my_system.positionSize.get_subsystem_position("SOFR").tail(5)

## 11. 组合构建与工具权重（Portfolios）

最后把各品种的子系统头寸按**品种权重 (instrument weights)** 加权，再乘**品种多样化乘数 (IDM)**：

$$\text{notional\_position} = \text{IDM} \times \sum_j w_j \cdot \text{subsystem\_position}_j$$

不配置时默认等权 + IDM=1。

In [ ]:
from systems.portfolio import Portfolios

portfolio = Portfolios()

# 固定品种权重（注意故意让权重之和 > 1，IDM 会处理）+ 固定 IDM
my_config.instrument_weights = dict(US10=.1, SOFR=.4, CORN=.3, SP500_micro=.8)
my_config.instrument_div_multiplier = 1.5
my_config.use_instrument_weight_estimates = False
my_config.use_instrument_div_mult_estimates = False

my_system = System([fcs, empty_rules, combiner, position_size, raw_data, portfolio], data, my_config)
my_system.portfolio.get_notional_position("SOFR").tail(5)

## 12. 完整账户盈亏统计（Account）

加入 `Account` 阶段后，可以结算整个组合的盈亏。此时成本（佣金 + 滑点，来自第 1 节看到的元数据）会被计入：

- `profits.percent` → **净收益**（扣除成本后）
- `profits.gross` → 毛收益
- `profits.costs` → 成本部分

In [ ]:
from systems.accounts.accounts_stage import Account

accounts = Account()
my_system = System([fcs, empty_rules, combiner, position_size, raw_data, portfolio, accounts],
                   data, my_config)
profits = my_system.accounts.portfolio()
profits.percent.stats()

In [ ]:
print("=== 净收益 ===");  print("Sharpe:", profits.percent.sharpe())
print("=== 毛收益 ===");  print("Sharpe:", profits.gross.percent.sharpe())
print("=== 成本拖累（年化） ===");  print("ann_mean:", profits.costs.percent.ann_mean())

profits.percent.curve().plot(figsize=(12, 4), title="Portfolio net % curve")

## 13. 使用预构建系统 simplesystem 与 futures_system

实际使用中**不会**手动拼装阶段，而是用预烘焙 (pre-baked) 入口函数，再按需替换 `data` 和 `config`：

- `simplesystem()`：对应前面手动搭建的例子，配置来自 `systems/provided/example/simplesystemconfig.yaml`
- `futures_system()`（`futures_chapter15`）：书中第 15 章的完整系统，分 `basesystem`（固定参数，快）和 `estimatedsystem`（滚动估计，慢）两个版本

In [ ]:
from systems.provided.example.simplesystem import simplesystem

my_system = simplesystem()
print(my_system)
my_system.portfolio.get_notional_position("SOFR").tail(5)

In [ ]:
# 传入自定义 config / data 的标准姿势（这是绝大多数时候创建系统的方式）
my_config = Config("systems.provided.example.simplesystemconfig.yaml")
my_data = csvFuturesSimData()
# ... 在这里按需修改 my_config / my_data ...
my_system = simplesystem(config=my_config, data=my_data)

# 完整的 chapter 15 系统（固定参数版，运行快）
from systems.provided.futures_chapter15.basesystem import futures_system
system = futures_system()
system.portfolio.get_notional_position("EUROSTX").tail(5)

## 14. 缓存系统计算结果（cache pickle/unpickle）

估计版系统（`estimatedsystem`）运行很慢。系统的中间计算结果都缓存在 `system.cache` 中，可以 pickle 到磁盘、跨会话复用：

```python
# 第一次（慢）：运行并保存缓存
system.cache.pickle("private.this_system_name.pck")

# 新会话：恢复缓存后，统计计算飞快
system.cache.unpickle("private.this_system_name.pck")
system.accounts.portfolio().sharpe()
```

> 下面这个 cell 演示机制（为避免长时间计算，只展示 pickle/unpickle 流程，不实际跑 estimatedsystem 的完整估计）。

---

## 学习路径总结与下一步

✅ 已完成：数据对象 → 单规则 → 盈亏 → 逐阶段构建完整系统 → 预烘焙系统 → 缓存

**下一步建议**：
1. 阅读 `docs/backtesting.md`（完整用户指南：默认值文件、成本模型、优化方法等）
2. 对照 `systems/provided/futures_chapter15/futuresconfig.yaml` 理解生产级配置
3. 深入源码：`systems/basesystem.py`（System 类与缓存机制）、`systems/stage.py`（阶段基类）
4. 之后再看生产环节：`sysproduction/`（数据更新、执行、风控）与 `sysbrokers/IB/`（IB 对接）

In [ ]:
# 缓存机制演示：basesystem 也有 cache；estimatedsystem 才真正需要它
import os
cache_file = "private/learning_demo_cache.pck"

system = futures_system()
system.accounts.portfolio().sharpe()   # 触发一些计算，填充缓存
system.cache.pickle(cache_file)        # 保存缓存
print("cache keys 示例:", list(system.cache.keys())[:5], "...")

# 模拟新会话：重建系统并恢复缓存
system2 = futures_system()
system2.cache.unpickle(cache_file)
print("恢复后 sharpe（秒出）:", system2.accounts.portfolio().sharpe())

os.remove(cache_file)  # 清理演示文件